In [1]:
import psycopg2
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")

conn = psycopg2.connect(DATABASE_URL)
conn.autocommit = True
cursor = conn.cursor()

print("Connected successfully")

Connected successfully


In [2]:
cursor.execute("CREATE EXTENSION IF NOT EXISTS vector;")
print("pgvector extension enabled")

cursor.execute("SELECT extversion FROM pg_extension WHERE extname = 'vector';")
version = cursor.fetchone()
print(f"pgvector version: {version[0] if version else 'not found'}")

pgvector extension enabled
pgvector version: 0.7.4


In [ ]:
# Create the chunks table

cursor.execute("""
    CREATE TABLE IF NOT EXISTS chunks (
        id SERIAL PRIMARY KEY,
        chunk_id VARCHAR(255) UNIQUE NOT NULL,
        acn VARCHAR(50),
        text TEXT NOT NULL,
        chunk_type VARCHAR(20),
        aircraft_model VARCHAR(255),
        flight_phase VARCHAR(255),
        mission VARCHAR(255),
        aircraft_operator VARCHAR(255),
        airport VARCHAR(50),
        state VARCHAR(50),
        report_year INTEGER,
        embedding vector(1024),
        created_at TIMESTAMP DEFAULT NOW()
    );
""")
print("Table 'chunks' created")

Table 'chunks' created


In [4]:
# Create the HNSW index for dense vector search

cursor.execute("""
    CREATE INDEX IF NOT EXISTS chunks_embedding_hnsw_idx
    ON chunks USING hnsw (embedding vector_cosine_ops)
    WITH (m = 16, ef_construction = 64);
""")
print("HNSW index created on embedding column")

HNSW index created on embedding column


In [5]:
# Create the full-text search index for keyword/BM25-style retrieval

cursor.execute("""
    ALTER TABLE chunks ADD COLUMN IF NOT EXISTS text_search_vector tsvector
    GENERATED ALWAYS AS (to_tsvector('english', text)) STORED;
""")

cursor.execute("""
    CREATE INDEX IF NOT EXISTS chunks_text_search_idx
    ON chunks USING GIN (text_search_vector);
""")
print("Full-text search column and GIN index created")

Full-text search column and GIN index created


In [6]:
cursor.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'chunks'
    ORDER BY ordinal_position;
""")

print("="*60)
print("CHUNKS TABLE SCHEMA")
print("="*60)
for col_name, data_type in cursor.fetchall():
    print(f"{col_name}: {data_type}")

cursor.execute("""
    SELECT indexname, indexdef FROM pg_indexes WHERE tablename = 'chunks';
""")
print("\n" + "="*60)
print("INDEXES ON CHUNKS TABLE")
print("="*60)
for idx_name, idx_def in cursor.fetchall():
    print(f"{idx_name}")

CHUNKS TABLE SCHEMA
id: integer
chunk_id: character varying
acn: character varying
text: text
chunk_type: character varying
aircraft_model: character varying
flight_phase: character varying
mission: character varying
aircraft_operator: character varying
airport: character varying
state: character varying
report_year: integer
embedding: USER-DEFINED
created_at: timestamp without time zone
text_search_vector: tsvector

INDEXES ON CHUNKS TABLE
chunks_pkey
chunks_chunk_id_key
chunks_embedding_hnsw_idx
chunks_text_search_idx


In [7]:
import numpy as np

test_embedding = np.random.rand(1024).tolist()

cursor.execute("""
    INSERT INTO chunks (chunk_id, acn, text, chunk_type, aircraft_model, embedding)
    VALUES (%s, %s, %s, %s, %s, %s)
    ON CONFLICT (chunk_id) DO NOTHING;
""", ('test_chunk_001', 'TEST001', 'Test narrative about engine vibration and bearing wear.', 'narrative', 'B737', test_embedding))

cursor.execute("SELECT chunk_id, text FROM chunks WHERE chunk_id = 'test_chunk_001';")
result = cursor.fetchone()
print(f"Test insert successful: {result}")

cursor.execute("""
    SELECT chunk_id, ts_rank(text_search_vector, query) AS rank
    FROM chunks, plainto_tsquery('english', 'engine vibration') query
    WHERE text_search_vector @@ query
    ORDER BY rank DESC LIMIT 5;
""")
print("\nFull-text search test results:")
for row in cursor.fetchall():
    print(row)

cursor.execute("DELETE FROM chunks WHERE chunk_id = 'test_chunk_001';")
print("\nTest row cleaned up")

Test insert successful: ('test_chunk_001', 'Test narrative about engine vibration and bearing wear.')

Full-text search test results:
('test_chunk_001', 0.09910322)

Test row cleaned up


In [8]:
cursor.close()
conn.close()
print("Connection closed. Schema setup complete.")

Connection closed. Schema setup complete.
